1: Import Dependencies

In [1]:
import os
import json
from zipfile import ZipFile
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

2026-03-01 13:09:34.590582: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 13:09:34.646657: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 13:09:35.975201: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


2: Set Seeds

In [2]:
import random
random.seed(0)

import numpy as np
np.random.seed(0)

import tensorflow as tf
tf.random.set_seed(0)

3: Dataset Path

In [3]:
# Dataset Path - LOCAL
base_dir = '/home/hp/Downloads/Projects/Backup/plant-disease-prediction-cnn-deep-leanring-project/Dataset'

# Verify the path
print(f"Base directory: {base_dir}")
print(f"Classes found: {os.listdir(base_dir)}")
print(f"Number of classes: {len(os.listdir(base_dir))}")

Base directory: /home/hp/Downloads/Projects/Backup/plant-disease-prediction-cnn-deep-leanring-project/Dataset
Classes found: ['Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Apple___Black_rot', 'Tomato_Late_blight', 'Apple___Cedar_apple_rust', 'Tomato__Tomato_mosaic_virus', 'Corn_(maize)___healthy', 'Potato___Late_blight', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Corn_(maize)___Northern_Leaf_Blight', 'Tomato_Leaf_Mold', 'Tomato__Target_Spot', 'Corn_(maize)___Common_rust_', 'Potato___healthy', 'Apple___Apple_scab', 'Tomato_Septoria_leaf_spot', 'Tomato_Bacterial_spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Early_blight', 'Tomato_healthy', 'Apple___healthy']
Number of classes: 23


4: Image Parameters

In [4]:
img_size = 224
batch_size = 32

5: Data Generators

In [5]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 28589 images belonging to 23 classes.
Found 7136 images belonging to 23 classes.


6: Model Architecture

In [6]:
# Model with GlobalAveragePooling2D (smaller model size)
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    #GlobalAveragePooling2D instead of Flatten
    layers.GlobalAveragePooling2D(),
    
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/home/hp/Downloads/Projects/Backup/plant-disease-prediction-cnn-deep-leanring-project/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-01 13:09:38.303106: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 23)             │         5,911 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 132,183 (516.34 KB)

 Trainable params: 132,183 (516.34 KB)

 Non-trainable params: 0 (0.00 B)

7: Training

In [8]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

Epoch 1/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 1026s 1s/step - accuracy: 0.3437 - loss: 2.1368 - val_accuracy: 0.6082 - val_loss: 1.2760
Epoch 2/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 992s 1s/step - accuracy: 0.6012 - loss: 1.2176 - val_accuracy: 0.6848 - val_loss: 0.9572
Epoch 3/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 966s 1s/step - accuracy: 0.6838 - loss: 0.9585 - val_accuracy: 0.7664 - val_loss: 0.7364
Epoch 4/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 955s 1s/step - accuracy: 0.7211 - loss: 0.8402 - val_accuracy: 0.7464 - val_loss: 0.7317
Epoch 5/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 956s 1s/step - accuracy: 0.7549 - loss: 0.7489 - val_accuracy: 0.8404 - val_loss: 0.4991
Epoch 6/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 953s 1s/step - accuracy: 0.7740 - loss: 0.6867 - val_accuracy: 0.8285 - val_loss: 0.5284
Epoch 7/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 926s 1s/step - accuracy: 0.7909 - loss: 0.6306 - val_accuracy: 0.8482 - val_loss: 0.4534
Epoch 8/10
894/894 ━━━━━━━━━━━━━━━━━━━━ 943s 1s/step - accuracy: 0.8099 - loss: 0.5754 - val_acc

8: Save Model

In [9]:
# Save model to app/trained_model folder
model.save('/home/hp/Downloads/Projects/Backup/plant-disease-prediction-cnn-deep-leanring-project/app/trained_model/plant_disease_prediction_model.h5')
print("Model saved successfully!")

Model saved successfully!


9: Save Class Indices

In [10]:
# Create a mapping from class indices to class names
class_indices = {v: k for k, v in train_generator.class_indices.items()}

# Save to app folder
json.dump(class_indices, open('/home/hp/Downloads/Projects/Backup/plant-disease-prediction-cnn-deep-leanring-project/app/class_indices.json', 'w'))
print("Class indices saved successfully!")

Class indices saved successfully!
